<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/Leave_Agent_Workshop_Notebook_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From a Question to an Agent
### Build an HR leave assistant, step by step (Gemini + LlamaIndex)

Student version: one function is left for you to complete in Step 3.

By the end you will have **one agent with three tools**:

| Tool | What it does | Backed by |
|---|---|---|
| `search_leave_policy` | Answers rule questions | The leave policy PDF (RAG) |
| `get_leave_balance` | Returns one person's balance | The leave database (SQL) |
| `get_leave_requests` | Lists one person's requests | The leave database (SQL) |

**The five steps**

| Step | Name | You will |
|---|---|---|
| 0 | Setup | Install, upload two files, add your Gemini key |
| 1 | Instruct | Write a clear system prompt |
| 2 | Ground | Read the policy PDF with RAG |
| 3 | Query | Build two specific SQL tools |
| 4 | Orchestrate | Give all three tools to an agent and test it |
| 5 | Protect | See the guardrails working |
| 6 | Ship | Put a chat window on top, then deploy to Hugging Face Spaces |

Run the cells from top to bottom. Each step adds one piece.

## Step 0. Setup

**0.1 Install the libraries** and check the versions.

In [1]:
!pip install -q llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-readers-file pypdf gradio

import llama_index.core, gradio
print("llama-index-core:", llama_index.core.__version__)
print("gradio:", gradio.__version__)

llama-index-core: 0.14.25
gradio: 6.31.0


**0.2 Upload the two files** you were given: the leave database (a `.db` file) and the leave policy (a `.pdf` file). Select both in the upload window.

In [ ]:
from google.colab import files

uploaded = files.upload()

DB_PATH = next((f for f in uploaded if f.lower().endswith((".db", ".sqlite"))), None)
PDF_PATH = next((f for f in uploaded if f.lower().endswith(".pdf")), None)
print("Database:", DB_PATH)
print("Policy PDF:", PDF_PATH)
assert DB_PATH and PDF_PATH, "Please upload both files: the .db database and the leave policy .pdf"

**0.3 Add your Gemini API key** to Colab Secrets. Never paste a key into the code.

1. Click the **key icon** in the left sidebar.
2. Add a secret named `GEMINI_API_KEY` and paste your key as the value.
3. Switch on **Notebook access**.

In [ ]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    raise SystemExit("Add GEMINI_API_KEY in Colab Secrets (key icon, left sidebar) and switch on Notebook access.")
print("Key loaded:", bool(GEMINI_API_KEY))

**0.4 Set up the model** and run a quick test. If your free tier hits rate limits, change `LLM_MODEL` to `"gemini-3.5-flash-lite"`.

In [ ]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

LLM_MODEL = "gemini-3.5-flash"
EMBED_MODEL = "gemini-embedding-001"

llm = GoogleGenAI(model=LLM_MODEL, api_key=GEMINI_API_KEY)
Settings.llm = llm
Settings.embed_model = GoogleGenAIEmbedding(model_name=EMBED_MODEL, api_key=GEMINI_API_KEY)

print(llm.complete("Say hello in five words.").text)

## Step 1. Instruct: clarity in, clarity out

A **system prompt** is the written job description you give the model before the conversation starts. A good one has five parts: **role, scope, rules, format, refusals**.

Run the same question twice: once with no instructions, once with a clear system prompt.

In [ ]:
from llama_index.core.llms import ChatMessage

question = "How many leave days do I have left?"

# A) No instructions
reply = llm.chat([ChatMessage(role="user", content=question)])
print("NO INSTRUCTIONS:\n", reply.message.content)

# B) A clear system prompt
CLEAR_PROMPT = """You are the HR leave assistant for Decoding Data Science (DDS).
Answer only questions about employee leave.
Never invent a leave balance. If you do not have the data, say so and ask for what you need.
If the employee's name is missing, ask for the full name.
Keep answers short."""

reply = llm.chat([ChatMessage(role="system", content=CLEAR_PROMPT), ChatMessage(role="user", content=question)])
print("\nCLEAR PROMPT:\n", reply.message.content)

**What to notice:** without instructions the model may invent a number. With a clear prompt it asks who you are and does not guess. This is still not enough, because the model has no data. The next steps give it documents and tools.

## Step 2. Ground: read the policy with RAG

**RAG** lets the model open the document and read the right passage before answering. Five moves: **load, chunk, embed, retrieve, answer**.

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

# 1. Load the PDF
documents = SimpleDirectoryReader(input_files=[PDF_PATH]).load_data()
print(len(documents), "page(s) loaded")

# 2-3. Chunk and embed (done together when the index is built)
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)
index = VectorStoreIndex.from_documents(documents)

# 4-5. Retrieve the top 3 passages and answer from them
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query("How much notice do I need for annual leave?")
print(response)

print("\nRetrieved passages (this is what the model actually read):")
for node in response.source_nodes:
    print(f"  score {node.score:.3f} | {node.text[:160]!r}")

**Chunking experiment.** How you split the document decides what the model can see. Compare two chunk sizes on the same question.

In [ ]:
q = "How much notice do I need for annual leave?"
for size in (128, 512):
    test_index = VectorStoreIndex.from_documents(
        documents,
        transformations=[SentenceSplitter(chunk_size=size, chunk_overlap=size // 8)],
    )
    chunks = len(test_index.docstore.docs)
    top = test_index.as_retriever(similarity_top_k=1).retrieve(q)[0]
    print(f"chunk_size={size}: {chunks} chunks | best passage: {top.text[:200]!r}\n")

**Habit to keep:** when an answer from a document is wrong, look at what was retrieved before blaming the model. Debug the retrieval, not the model.

## Step 3. Query: two specific SQL tools

First, look at the database. We open it **read-only**, so nothing here can change it.

In [ ]:
import sqlite3
import pandas as pd
from contextlib import closing

def connect():
    # Read-only connection: even a bug cannot change the data
    return sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

def read_sql(sql, params=()):
    with closing(connect()) as con:
        return pd.read_sql_query(sql, con, params=params)

tables = read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view')")
display(tables)

for name in tables["name"]:
    print("\n" + name)
    display(read_sql(f"SELECT * FROM {name} LIMIT 3"))

You should see three objects: **employees** (who works here), **leave_requests** (every request) and **leave_balances** (a view that computes used and remaining days from the requests).

### Why specific tools, not free SQL?

| Free SQL | Specific tools (ours) |
|---|---|
| The model writes any query | The model only supplies a name |
| Can guess wrong column names or run heavy queries | One fixed, parameterised query per tool |
| Needs heavy guardrails | Read-only connection and fixed queries |

First a small helper that finds the employee, and handles the two awkward cases: **not found** and **several matches**.

In [ ]:
def name_column(con):
    cols = [r[1] for r in con.execute("PRAGMA table_info(employees)")]
    return "full_name" if "full_name" in cols else "name"

def resolve_employee(con, employee_name):
    """Return (employee, error_message)."""
    col = name_column(con)
    wanted = employee_name.strip()
    rows = con.execute(
        f"SELECT employee_id, {col}, department FROM employees WHERE {col} LIKE ? ORDER BY {col} LIMIT 6",
        (f"%{wanted}%",),
    ).fetchall()
    if not rows:
        return None, f"No employee found matching '{employee_name}'."
    exact = [r for r in rows if r[1].lower() == wanted.lower()]
    if len(rows) > 1 and not exact:
        options = "; ".join(f"{r[1]} ({r[2]})" for r in rows)
        return None, f"Several employees match '{employee_name}': {options}. Ask which one is meant."
    return (exact[0] if exact else rows[0]), None

### Tool 2: `get_leave_balance`
The model passes a name. The query inside is fixed.

In [ ]:
def get_leave_balance(employee_name: str) -> str:
    """Return one employee's leave balance (entitlement, used, remaining, pending).
    Input: the employee's full name."""
    with closing(connect()) as con:
        emp, error = resolve_employee(con, employee_name)
        if error:
            return error
        try:
            cur = con.execute("SELECT * FROM leave_balances WHERE employee_id = ?", (emp[0],))
        except sqlite3.OperationalError:      # a database without the leave_balances view
            cur = con.execute("SELECT * FROM employees WHERE employee_id = ?", (emp[0],))
        row = cur.fetchone()
        if row is None:
            return f"No balance record found for {emp[1]}."
        cols = [d[0] for d in cur.description]
        lines = [f"- {c}: {v}" for c, v in zip(cols, row) if c != "employee_id"]
        return f"Leave balance for {emp[1]} ({emp[2]}):\n" + "\n".join(lines)

print(get_leave_balance("Aisha Khan"))

### Tool 3: `get_leave_requests` (your turn)
Use the same pattern as the balance tool. The model passes a name and an optional status such as `Pending`. The five steps are in the comments.

Stuck? Open the solution below after you have tried.

<details><summary>Solution (open after you try)</summary>

```python
def get_leave_requests(employee_name: str, status: str = "") -> str:
    with closing(connect()) as con:
        emp, error = resolve_employee(con, employee_name)
        if error:
            return error
        sql = "SELECT * FROM leave_requests WHERE employee_id = ?"
        params = [emp[0]]
        if status and status.strip():
            sql += " AND LOWER(status) = LOWER(?)"
            params.append(status.strip())
        sql += " ORDER BY start_date DESC LIMIT 15"
        cur = con.execute(sql, params)
        rows = cur.fetchall()
        if not rows:
            suffix = f" with status '{status}'" if status and status.strip() else ""
            return f"No leave requests found for {emp[1]}{suffix}."
        cols = [d[0] for d in cur.description]
        keep = [i for i, c in enumerate(cols) if c not in ("employee_id", "request_id")]
        lines = [", ".join(f"{cols[i]}={r[i]}" for i in keep) for r in rows]
        return f"Leave requests for {emp[1]} ({emp[2]}):\n" + "\n".join(f"- {l}" for l in lines)
```

</details>

In [ ]:
def get_leave_requests(employee_name: str, status: str = "") -> str:
    """List one employee's leave requests with type, dates, working days and status.
    Input: the employee's full name and an optional status (Approved, Pending, Rejected)."""
    with closing(connect()) as con:
        emp, error = resolve_employee(con, employee_name)
        if error:
            return error
        # YOUR TURN. Follow the five steps, then run the cell.
        # 1. Start with:  sql = "SELECT * FROM leave_requests WHERE employee_id = ?"   and   params = [emp[0]]
        # 2. If a status was given, add:  AND LOWER(status) = LOWER(?)  to sql, and append the status to params
        # 3. Finish the query with:  ORDER BY start_date DESC LIMIT 15   then run it with  con.execute(sql, params)
        # 4. If there are no rows, return a clear "No leave requests found" message
        # 5. Otherwise return one line per request, starting with the employee name
        raise NotImplementedError("Complete get_leave_requests, then run the cells below")

**Test both tools directly, before any agent is involved.** If a tool is wrong here, the agent will be wrong too.

In [ ]:
print(get_leave_requests("John Doe"))
print()
print(get_leave_requests("John Doe", "Pending"))
print()
print(get_leave_balance("Nobody Here"))

## Step 4. Orchestrate: three tools, one agent

The agent never sees your function bodies. It sees a **name and a description**, and uses only those to decide which tool to call. So the description is the routing logic. Notice the "Do NOT use for..." wording in the first tool.

In [ ]:
from llama_index.core.tools import FunctionTool, QueryEngineTool
from llama_index.core.agent.workflow import FunctionAgent, ToolCall

policy_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="search_leave_policy",
    description=(
        "Answers questions about leave RULES and entitlements from the DDS leave policy "
        "(notice periods, approvals, limits). Do NOT use for any person's balance or requests."
    ),
)
balance_tool = FunctionTool.from_defaults(
    fn=get_leave_balance,
    name="get_leave_balance",
    description=(
        "Returns one employee's leave balance: entitlement, used, remaining, pending. "
        "Input: the employee's full name."
    ),
)
requests_tool = FunctionTool.from_defaults(
    fn=get_leave_requests,
    name="get_leave_requests",
    description=(
        "Lists one employee's leave requests with type, dates, working days and status. "
        "Input: full name and an optional status filter (Approved, Pending, Rejected)."
    ),
)

SYSTEM_PROMPT = """You are the HR leave assistant for Decoding Data Science (DDS).
Answer only questions about employee leave.

Rules:
- Rules, entitlements, notice periods and approvals come from search_leave_policy.
- A person's balance comes from get_leave_balance. A person's requests come from get_leave_requests.
- Never invent or estimate a number. If a tool does not return it, say so.
- If a question needs both the policy and personal data, call both tools before answering.
- If the employee's name is missing, ask for the full name. If several employees match, ask which one.
- If the policy does not cover something, say that the policy does not cover it.
- You are read-only. You cannot approve, submit, change or cancel leave. Refuse politely and point to the line manager or HR.
- Keep answers short and state numbers clearly, for example: 2 of 25 annual days remaining."""

agent = FunctionAgent(
    tools=[policy_tool, balance_tool, requests_tool],
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
)
print("Agent ready with tools:", [t.metadata.name for t in agent.tools])

A helper that runs the agent and shows **which tools it called**. This is the ReAct loop: think, act, observe, answer.

In [ ]:
async def ask(question, show=True):
    handler = agent.run(question)
    used = []
    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            used.append(event.tool_name)
            if show:
                print(f"  tool call -> {event.tool_name}({event.tool_kwargs})")
    response = await handler
    return str(response), used

answer, used = await ask("Can Aisha Khan take 12 days off in December?")
print("\nANSWER:", answer)

### The test set
Before you run it, **predict** which tool or tools the agent will call for each question. The last column is the expected answer. A good agent picks the right tools every time.

In [ ]:
import asyncio

TESTS = [
    ("How much notice do I need for annual leave?",             {"search_leave_policy"}),
    ("How many annual days does Aisha Khan have left?",         {"get_leave_balance"}),
    ("Show John Doe's leave requests.",                         {"get_leave_requests"}),
    ("Can Aisha Khan take 12 days off in December?",            {"get_leave_balance", "search_leave_policy"}),
    ("Is John Doe's pending request within the notice rules?",  {"get_leave_requests", "search_leave_policy"}),
    ("Which of Aisha Khan's requests were rejected?",           {"get_leave_requests"}),
    ("What is the probation period?",                           {"search_leave_policy"}),
    ("Approve my leave request.",                               set()),
    ("How many days does Sam Smith have left?",                 {"get_leave_balance"}),
]

passed = 0
for question, expected in TESTS:
    answer, used = await ask(question, show=False)
    ok = set(used) == expected
    passed += ok
    print(("PASS" if ok else "CHECK"), "|", question)
    print("     expected:", sorted(expected) or "no tool", "| used:", sorted(set(used)) or "no tool")
    print("     answer:", answer[:220].replace("\n", " "), "\n")
    await asyncio.sleep(2)      # be kind to free-tier rate limits

print(f"Tool choice matched on {passed} of {len(TESTS)} questions")

A **CHECK** is not always a failure. Read the answer. Sometimes the agent used an extra tool but answered well, and sometimes the test expectation needs fixing. Reading these traces is how you improve an agent.

Want to see how much the description matters? Change the first tool's description to just `"Leave information"`, rebuild the agent, and run the test set again.

## Step 5. Protect: fail safely

| Guardrail | Where it lives |
|---|---|
| Read-only database | `mode=ro` in `connect()` |
| The model never writes SQL | The tools take a name, the query is fixed |
| Employee not found | `resolve_employee` returns a clear message |
| Ambiguous name | `resolve_employee` lists the matches |
| Out of scope or write requests | System prompt, and there are no write tools |
| Not in the policy | System prompt rule |

Prove the first one. This delete must fail:

In [ ]:
try:
    connect().execute("DELETE FROM employees")
    print("Delete worked. That should never happen.")
except Exception as e:
    print("Blocked as expected:", e)

Now try to break the agent yourself. Ask something outside leave, ask for a salary, or ask it to delete a request.

In [ ]:
answer, used = await ask("Please delete John Doe's pending request.")
print(answer)

## Step 6. Ship: put a chat window on top

Gradio gives us a chat interface in a few lines. It also shows which tools were used under each answer.

In [ ]:
import gradio as gr

async def chat(message, history):
    try:
        answer, used = await ask(message, show=False)
    except Exception as e:
        return f"Error: {e}"
    return f"{answer}\n\n---\n*Tools used: {', '.join(dict.fromkeys(used)) or 'none'}*"

demo = gr.ChatInterface(
    fn=chat,
    title="DDS Leave Assistant",
    description="One agent, three tools. Ask with a full employee name, for example: Aisha Khan.",
    examples=[
        "How much notice do I need for annual leave?",
        "How many annual days does Aisha Khan have left?",
        "Show John Doe's pending leave requests.",
        "Can Aisha Khan take 12 days off in December?",
    ],
    cache_examples=False,
)
demo.launch(share=True)

## Step 7. Deploy to Hugging Face Spaces

The same code runs as a Space. You need four things in the Space:

```
app.py              the agent and the chat window (the code above, in one file)
requirements.txt    the libraries
README.md           the Space settings
data/               your .db file and the leave policy .pdf
```

Add `GEMINI_API_KEY` as a **secret** in the Space settings. The key never goes in the code.

## What comes next
1. Memory and identity across turns, so the agent knows who "my" refers to.
2. Free-form SQL for analysts, with guardrails.
3. Write tools, such as submitting a request, with human approval.
4. Automatic evaluation of tool choice on a larger test set.

Take it further with the **AI Accelerator Bootcamp** at decodingdatascience.com.